## Structures

In [1]:
def preproc_csv_out(content: str, fill_none: bool = False) -> str:
    if fill_none and content is None:
        return ""
    if type(content) is not str:
        return content
    if "," in content:
        if '"' in content:
            content = content.replace('"', '""')
        content = f'"{content}"'
    return content\
        .replace("\n", "\\n")\
        .replace("\r", " ")\
        .strip("\r\n")

In [2]:
_ = preproc_csv_out

class Drug:
    def __init__(self,
        drugbank_id: str,
        name: str,
        description: str,
        cas_number: str,
        state: str,
        unii: str,
    ):
        self.drugbank_id = drugbank_id
        self.name = name
        self.description = description
        self.cas_number = cas_number
        self.state = state
        self.unii = unii
    
    def __str__(self):
        return f"Drug({self.drugbank_id}, {self.name}, {self.description}, {self.cas_number}, {self.state}, {self.unii})"

    def __repr__(self):
        return self.__str__()
    
    def to_csv(self):
        return f"{_(self.drugbank_id)},{_(self.name)},{_(self.description)},{_(self.cas_number)},{_(self.state)},{_(self.unii)}"


class DerivedFromDrug:
    def __init__(self, drug: Drug=None):
        self.drug = drug

    @property
    def derived_from(self):
        if self.drug is not None:
            return self.drug.drugbank_id
        return None


class Category:
    def __init__(self,
        id: str,
        category: str
    ):
        self.id = id
        self.category = category

    def __str__(self):
        return f"Category({self.id}, {self.category})"

    def __repr__(self):
        return self.__str__()
    
    def to_csv(self):
        return f"{_(self.id)},{_(self.category)}"

class ExternalDatabaseIdentifier(DerivedFromDrug):
    def __init__(self,
        external_id: str,
        resource: str,
        drug: Drug = None
    ):
        super().__init__(drug)
        self.external_id = external_id
        self.resource = resource

    def __str__(self):
        return f"ExternalDatabaseIdentifier({self.external_id}, {self.derived_from}, {self.resource})"

    def __repr__(self):
        return self.__str__()

    def to_csv(self):
        return f"{_(self.external_id)},{_(self.derived_from)},{_(self.resource)}"

class Product(DerivedFromDrug):
    def __init__(self,
        name: str,
        labeller: str,
        strength: str,
        route: str,
        approved: bool,
        ndc_product_code: str,
        ema_ma_number: str,
        drug: Drug = None,
    ):
        super().__init__(drug)
        self.name = name
        self.labeller = labeller
        self.strength = strength
        self.route = route
        self.approved = approved
        self.ndc_product_code = ndc_product_code
        self.ema_ma_number = ema_ma_number

    def __str__(self):
        return f"Product({self.derived_from}, {self.name}, {self.labeller}, {self.strength}, {self.route}, {self.approved}, {self.ndc_product_code}, {self.ema_ma_number})"
        
    def __repr__(self):
        return self.__str__()

    def to_csv(self):
        return f"{_(self.derived_from)},{_(self.name)},{_(self.labeller)},{_(self.strength)},{_(self.route)},{_(self.approved)},{_(self.ndc_product_code)},{_(self.ema_ma_number)}"

class ExperimentalProperty(DerivedFromDrug):
    def __init__(self,
        kind: str,
        value: str,
        source: str,
        drug: Drug = None
    ):
        super().__init__(drug)
        self.kind = kind
        self.value = value
        self.source = source

    def __str__(self):
        return f"ExperimentalProperty({self.derived_from}, {self.kind}, {self.value}, {self.source})"

    def __repr__(self):
        return self.__str__()

    def to_csv(self):
        return f"{_(self.derived_from)},{_(self.kind)},{_(self.value)},{_(self.source)}"

class Target:
    def __init__(self,
        id: str,
        name: str,
        organism: str,
    ):
        self.id = id
        self.name = name
        self.organism = organism
    
    def __str__(self):
        return f"Target({self.id}, {self.name}, {self.organism})"

    def __repr__(self):
        return self.__str__()

    def to_csv(self):
        return f"{_(self.id)},{_(self.name)},{_(self.organism)}"
    
class Reference:
    def __init__(self,
        ref_id: str,
        pubmed_id: str,
        citation: str
    ):
        self.ref_id = ref_id
        self.pubmed_id = pubmed_id
        self.citation = citation
    
    def __str__(self):
        return f"Reference({self.ref_id}, {self.pubmed_id}, {self.citation})"

    def __repr__(self):
        return self.__str__()
    
    def to_csv(self):
        return f"{_(self.ref_id)},{_(self.pubmed_id)},{_(self.citation)}"

## XML

Drugbank's FULL XML format database is required. Download it and place in the parent directory or modify the below cell.

```py
tree = ET.parse('../drugbank.xml')
```

In [3]:
import xml.etree.ElementTree as ET

TAG_NAME_PREFIX = "{http://www.drugbank.ca}"

In [4]:
tree = ET.parse("../full_database.xml")

### Processing functions


In [5]:
def proc_drug(element):
    drugbank_id = None
    for each in element.findall(f"{TAG_NAME_PREFIX}drugbank-id"):
        isprimary = each.attrib.get("primary")
        if isprimary and isprimary == "true":
            drugbank_id = each.text
    name = element.find(f"{TAG_NAME_PREFIX}name")
    description = element.find(f"{TAG_NAME_PREFIX}description")
    cas_number = element.find(f"{TAG_NAME_PREFIX}cas-number")
    state = element.find(f"{TAG_NAME_PREFIX}state")
    unii = element.find(f"{TAG_NAME_PREFIX}unii")
    return Drug(
        drugbank_id=drugbank_id if drugbank_id is not None else "",
        name=name.text if name is not None and name.text is not None else "",
        description=description.text if description is not None and description.text is not None else "",
        cas_number=cas_number.text if cas_number is not None and cas_number.text is not None else "",
        state=state.text if state is not None and state.text is not None else "",
        unii=unii.text if unii is not None and unii.text is not None else "",
    )

In [6]:
def proc_category(element: ET.Element):
    """
    <categories>
        <category>
            <category>Anticoagulants</category>
            <mesh-id>D000925</mesh-id>
        </category>
        ...
    </categories>
    """
    if element.tag != f"{TAG_NAME_PREFIX}categories":
        raise ValueError("Invalid element for categories")
    categories = []
    for category in element:
        if category.tag != f"{TAG_NAME_PREFIX}category":
            raise ValueError("Invalid element for categories")
    
        mesh_id = category.find(f"{TAG_NAME_PREFIX}mesh-id")
        category = category.find(f"{TAG_NAME_PREFIX}category")

        categories.append(Category(
            id=mesh_id.text if mesh_id is not None and mesh_id.text is not None else '',
            category=category.text if category is not None and category.text is not None else ''
        ))
    return categories

In [7]:
def proc_external_identifier(element: ET.Element):
    """
    <external-identifiers>
        <external-identifier>
            <resource>Drugs Product Database (DPD)</resource>
            <identifier>11916</identifier>
        </external-identifier>
        <external-identifier>
        ...
    </external-identifiers>
    """
    identifiers = []
    if element.tag != f"{TAG_NAME_PREFIX}external-identifiers":
        raise ValueError("Invalid element for external identifiers")
    for external_identifier in element:
        if external_identifier.tag != f"{TAG_NAME_PREFIX}external-identifier":
            raise ValueError("Invalid element for external identifiers: " + external_identifier.tag)
        resource = external_identifier.find(f"{TAG_NAME_PREFIX}resource")
        identifier = external_identifier.find(f"{TAG_NAME_PREFIX}identifier")
        identifiers.append(ExternalDatabaseIdentifier(
            external_id=identifier.text if identifier is not None else '',
            resource=resource.text if resource is not None else ''
        ))
    return identifiers

In [8]:
def proc_products(element: ET.Element):
    if element.tag != f"{TAG_NAME_PREFIX}products":
        raise ValueError("Invalid element for products")
    products = []
    for each in element:
        if each.tag != f"{TAG_NAME_PREFIX}product":
            raise ValueError("Invalid element for products")
        name = each.find(f"{TAG_NAME_PREFIX}name")
        labeller = each.find(f"{TAG_NAME_PREFIX}labeller")
        strength = each.find(f"{TAG_NAME_PREFIX}strength")
        route = each.find(f"{TAG_NAME_PREFIX}route")
        approved = each.find(f"{TAG_NAME_PREFIX}approved")
        ndc_product_code = each.find(f"{TAG_NAME_PREFIX}ndc-product-code")
        ema_ma_number = each.find(f"{TAG_NAME_PREFIX}ema-ma-number")
        products.append(Product(
            name=name.text if name is not None else '',
            labeller=labeller.text if labeller is not None and labeller.text is not None else '',
            strength=strength.text if strength is not None and strength.text is not None else '',
            route=route.text if route is not None and route.text is not None else '',
            approved=approved.text.lower() == 'true' if approved is not None else False,
            ndc_product_code=ndc_product_code.text if ndc_product_code is not None and ndc_product_code.text is not None else '',
            ema_ma_number=ema_ma_number.text if ema_ma_number is not None and ema_ma_number.text is not None else ''
        ))
    return products

In [9]:
def proc_experimental_properties(element: ET.Element):
    if element.tag != f"{TAG_NAME_PREFIX}experimental-properties":
        raise ValueError("Invalid element for experimental properties")
    experimentals = []
    for each in element:
        if each.tag != f"{TAG_NAME_PREFIX}property":
            raise ValueError("Invalid element for experimental properties")
        kind = each.find(f"{TAG_NAME_PREFIX}kind")
        value = each.find(f"{TAG_NAME_PREFIX}value")
        source = each.find(f"{TAG_NAME_PREFIX}source")
        experimentals.append(ExperimentalProperty(
            kind=kind.text if kind is not None and kind.text is not None else '',
            value=value.text if value is not None and value.text is not None else '',
            source=source.text if source is not None and source.text is not None else ''
        ))
    return experimentals

In [10]:
reference_types = [
    ["articles", "article"],
    ["textbooks", "textbook"],
    ["links", "link"],
    ["attachments", "attachment"],
]

_wrapper_tags, _element_tags = [], []
for x in reference_types:
    _wrapper_tags.append(f"{TAG_NAME_PREFIX}{x[0]}")
    _element_tags.append(f"{TAG_NAME_PREFIX}{x[1]}")

def proc_references(element: ET.Element):
    if element.tag != f"{TAG_NAME_PREFIX}references":
        raise ValueError("Invalid element for references")
    references = []
    for articles in element:
        if articles.tag not in _wrapper_tags:
            raise ValueError("Invalid element for references")
        for article in articles:
            if article.tag not in _element_tags:
                raise ValueError("Invalid element for references")
            ref_id = article.find(f"{TAG_NAME_PREFIX}ref-id")
            pubmed_id = article.find(f"{TAG_NAME_PREFIX}pubmed-id")
            citation = article.find(f"{TAG_NAME_PREFIX}citation")
            references.append(Reference(
                ref_id=ref_id.text if ref_id is not None and ref_id.text is not None else '',
                pubmed_id=pubmed_id.text if pubmed_id is not None and pubmed_id.text is not None else '',
                citation=citation.text if citation is not None and citation.text is not None else ''
            ))
    return references

def proc_targets(element: ET.Element):
    if element.tag != f"{TAG_NAME_PREFIX}targets":
        raise ValueError("Invalid element for targets")
    targets = []
    references = []
    for each in element:
        if each.tag != f"{TAG_NAME_PREFIX}target":
            raise ValueError("Invalid element for targets")
        id = each.find(f"{TAG_NAME_PREFIX}id")
        name = each.find(f"{TAG_NAME_PREFIX}name")
        organism = each.find(f"{TAG_NAME_PREFIX}organism")
        targets.append(Target(
            id=id.text if id is not None and id.text is not None else '',
            name=name.text if name is not None and name.text is not None else '',
            organism=organism.text if organism is not None and organism.text is not None else ''
        ))
        references_element = each.find(f"{TAG_NAME_PREFIX}references")
        _references = proc_references(references_element)
        references.extend(_references)
    return targets, references

### Apply processing functions

- 식별자가 누락된 카테고리 값이 존재합니다. 아래의 `CATEGORY_IDENTIFIER_NULL_AUTOFILL`을 `True`로 지정하면, 이후에 선언한 전처리 함수가 식별자를 자체적으로 부여합니다. 이렇게 부여된 식별자는 `A`로 시작하는 문자열입니다.

- 문제 요구사항 외에도 이후에 데이터베이스에서 관계를 다루는 데 사용할 수 있도록, `drug`의 멤버로 존재한 객체에는 `derived-from` 속성을 추가하였습니다. `derived-from`은 `drug`의 식별자인 `drugbank-id`입니다.

In [11]:
CATEGORY_IDENTIFIER_NULL_AUTOFILL = True

In [12]:
root = tree.getroot()

In [13]:
drugs = []
categories = []
external_identifiers = []
products = []
experimental_properties = []
targets = []
references = []

drug_category_relations = []
drug_target_relations = []
drug_reference_relations = []

unique_validation_dict = { key: set() for key in [
    "Drugs",
    "Categories",
    "DrugCategoryRelations",
    "ExternalDatabaseIdentifiers",
    "Products",
    "ExperimentalProperties",
    "Targets",
    "DrugTargetRelations",
    "References",
    "DrugReferenceRelations"
]}

def is_unique(_type: str, value: object) -> bool:
    """
    모든 필드가 동일한 중복 행이 존재하는지 확인합니다.
    앞서 이 값을 통과한 값 중 value의 csv 표현이 완전히 동일한 값이 있는지 확인합니다.
    이 함수는 csv 표현을 .to_csv() 메서드를 통해 획득합니다. 따라서 value는 .to_csv()
    메서드를 구현한 객체여야 합니다.

    *이 함수는 unique_validation_dict에 의존하고, 검증 대상이 중복 여부와 관계 없이
    항상 이 함수를 통과함을 가정하고 작성되었습니다. 만약 두 조건이 충족되지 않는다면, 검증은
    잘못 수행될 가능성이 높습니다.
    """
    if _type not in unique_validation_dict:
        raise ValueError(f"Unknown type: {_type}")
    value = value.to_csv() if hasattr(value, 'to_csv') else str(value)
    if value in unique_validation_dict[_type]:
        return False
    unique_validation_dict[_type].add(value)
    return True

category_identifier_auto_increment = 0
def fill_category_identifier(category: Category):
    global category_identifier_auto_increment
    if category.id is None or category.id == '':
        category.id = f"A{category_identifier_auto_increment:06d}"
        category_identifier_auto_increment += 1

def apply_drug(obj: object, drug: Drug):
    if not isinstance(obj, DerivedFromDrug):
        raise ValueError("Object must be an instance of DerivedFromDrug")
    obj.drug = drug
    return obj

for drug_element in root:
    if drug_element.tag == f"{TAG_NAME_PREFIX}drug":
        _drug = proc_drug(drug_element)
        drugs.append(_drug)

        _categories = proc_category(drug_element.find(f"{TAG_NAME_PREFIX}categories"))
        for category in _categories:
            if CATEGORY_IDENTIFIER_NULL_AUTOFILL:
                if category.id is None or category.id == '':
                    fill_category_identifier(category)
            if is_unique("Categories", value=category.id):
                categories.append(category)
            
            drug_category_relation = (_drug.drugbank_id, category.id)
            if is_unique("DrugCategoryRelations", drug_category_relation):
                drug_category_relations.append(drug_category_relation)
        
        _external_identifiers = proc_external_identifier(drug_element.find(f"{TAG_NAME_PREFIX}external-identifiers"))
        for external_identifier in _external_identifiers:
            if is_unique("ExternalDatabaseIdentifiers", value=external_identifier.external_id):
                external_identifiers.append(
                    apply_drug(external_identifier, _drug)
                )

        """
        > 5-3. 모든 필드가 동일한 중복 행이 발생할 수 있으므로, 이러한 중복 행은 제거하여 데이터 무결성을 유지해야 한다.
        """
        _products = proc_products(drug_element.find(f"{TAG_NAME_PREFIX}products"))
        for product in _products:
            if is_unique("Products", value=product):
                apply_drug(product, _drug)
                products.append(product)
        
        _experimental_properties = proc_experimental_properties(drug_element.find(f"{TAG_NAME_PREFIX}experimental-properties"))
        experimental_properties.extend(
            map(lambda x: apply_drug(x, _drug), _experimental_properties)
        )

        targets_element = drug_element.find(f"{TAG_NAME_PREFIX}targets")
        _targets, _references = proc_targets(targets_element)
        for target in _targets:
            if is_unique("Targets", value=target):
                targets.append(target)
            
            drug_target_relation = (_drug.drugbank_id, target.id)
            if is_unique("DrugTargetRelations", drug_target_relation):
                drug_target_relations.append(drug_target_relation)

        for reference in _references:
            if is_unique("References", value=reference.ref_id):
                references.append(reference)
            
            drug_reference_relation = (_drug.drugbank_id, reference.ref_id)
            if is_unique("DrugReferenceRelations", drug_reference_relation):
                drug_reference_relations.append(drug_reference_relation)


## Output

In [14]:
%mkdir -p outputs

In [15]:
headers = {
    "Drugs": ["drugbank-id", "name", "description", "cas-number", "state", "unii"],
    "Categories": ["mesh-id", "category"],
    "ExternalDatabaseIdentifiers": ["identifier", "derived-from", "resource"],
    "Products": ["derived-from", "name", "labeller", "strength", "route", "approved", "ndc-product-code", "ema-ma-number"],
    "ExperimentalProperties": ["derived-from", "kind", "value", "source"],
    "Targets": ["id", "derived-from", "name", "organism"],
    "References": ["ref-id", "derived-from", "pubmed-id", "citation"],
}

tables = {
    "Drugs": drugs,
    "Categories": categories,
    "ExternalDatabaseIdentifiers": external_identifiers,
    "Products": products,
    "ExperimentalProperties": experimental_properties,
    "Targets": targets,
    "References": references
}

for table in tables:
    with open(f"./outputs/{table}.csv", "w", encoding="utf-8") as f:
        f.write(",".join(headers[table]) + "\n")
        for each in tables[table]:
            f.write(each.to_csv() + "\n")
    print(f"Table {table} written to {table}.csv")

Table Drugs written to Drugs.csv
Table Categories written to Categories.csv
Table ExternalDatabaseIdentifiers written to ExternalDatabaseIdentifiers.csv
Table Products written to Products.csv
Table ExperimentalProperties written to ExperimentalProperties.csv
Table Targets written to Targets.csv
Table References written to References.csv


In [16]:
headers = {
    "DrugCategoryRelations": ["drug", "category"],
    "DrugTargetRelations": ["drug", "target"],
    "DrugReferenceRelations": ["drug", "reference"]
}
tables = {
    "DrugCategoryRelations": drug_category_relations,
    "DrugTargetRelations": drug_target_relations,
    "DrugReferenceRelations": drug_reference_relations
}

for table in tables:
    with open(f"./outputs/{table}.csv", "w", encoding="utf-8") as f:
        f.write(",".join(headers[table]) + "\n")
        for each in tables[table]:
            f.write(f"{each[0]},{each[1]}\n")
    print(f"Table {table} written to {table}.csv")

Table DrugCategoryRelations written to DrugCategoryRelations.csv
Table DrugTargetRelations written to DrugTargetRelations.csv
Table DrugReferenceRelations written to DrugReferenceRelations.csv


## Validation

In [17]:
from enum import Enum

class AnomalyType(Enum):
    LENGTH_NOT_EQUAL_TO_HEADER = 1
    ROW_IS_ALL_NONE_OR_EMPTY = 2
    SOME_COLUMNS_ARE_NONE_OR_EMPTY = 3

In [18]:
import csv

targets = ["Drugs", "Categories", "ExternalDatabaseIdentifiers", "Products", "ExperimentalProperties", "Targets", "References"]
anomalies = { each: [] for each in targets }

for each in targets:
    with open(f"./outputs/{each}.csv", "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) != len(headers[each]):
                anomalies[each].append({"type": AnomalyType.LENGTH_NOT_EQUAL_TO_HEADER, "row": row})
            _column_is_ne_each_row = [_ is None or _ == "" for _ in row]
            if all(_column_is_ne_each_row):
                anomalies[each].append({"type": AnomalyType.ROW_IS_ALL_NONE_OR_EMPTY, "row": row})
            elif any(_column_is_ne_each_row):
                anomalies[each].append({"type": AnomalyType.SOME_COLUMNS_ARE_NONE_OR_EMPTY, "row": row})

KeyError: 'Drugs'